# Plataforma de Inteligência da Saúde Infantil

Camada complementar ao sistema relacional existente. O sistema de origem envia eventos por HTTP/JSON; esta plataforma valida, mantém projeções mínimas no MongoDB e disponibiliza inteligência operacional, monitoramento e resumos de alta. Não acessa nem substitui o SQL Server.

**Ambiente:** Google Colab para desenvolvimento e demonstração, somente com dados sintéticos. Não use dados reais de pacientes no Colab.

In [ ]:
# Célula 2 — Dependências
# Execute no Google Colab. Dependências de túnel serão preparadas na célula 11.
%pip -q install "pymongo[srv]" fastapi uvicorn streamlit pandas plotly pydantic reportlab requests httpx pytest mongomock

In [ ]:
# Célula 3 — Credenciais
# No Atlas: Connect > Drivers > Python. Use uma URI de usuário com privilégios mínimos.
# A URI, a chave de integração e a senha da interface são solicitadas sem eco e não são gravadas em arquivos.
from getpass import getpass
from pathlib import Path
import os

MONGO_URI = getpass("MongoDB Atlas URI: ").strip()
DATABASE_NAME = "inteligencia_saude_infantil"
API_INTEGRATION_KEY = getpass("Chave exclusiva da API de integração (mínimo 24 caracteres): ").strip()
APP_ADMIN_USERNAME = input("Usuário administrador da interface: ").strip()
APP_ADMIN_PASSWORD = getpass("Senha inicial do administrador (mínimo 12 caracteres): ")

if not MONGO_URI.startswith(("mongodb+srv://", "mongodb://")):
    raise ValueError("Informe uma URI MongoDB Atlas válida.")
if len(API_INTEGRATION_KEY) < 24:
    raise ValueError("A chave de integração precisa ter ao menos 24 caracteres.")
if not APP_ADMIN_USERNAME or len(APP_ADMIN_PASSWORD) < 12:
    raise ValueError("Informe usuário e senha de administrador com ao menos 12 caracteres.")

os.environ.update({
    "MONGO_URI": MONGO_URI,
    "MONGO_DATABASE": DATABASE_NAME,
    "API_INTEGRATION_KEY": API_INTEGRATION_KEY,
    "APP_ADMIN_USERNAME": APP_ADMIN_USERNAME,
    "APP_ADMIN_PASSWORD": APP_ADMIN_PASSWORD,
})
APP_HOME = Path.cwd() / "plataforma_saude_infantil_app"
APP_HOME.mkdir(parents=True, exist_ok=True)
os.environ["APP_HOME"] = str(APP_HOME)
print("Credenciais mantidas apenas na memória desta sessão.")

In [ ]:
# Célula 4 — Conexão MongoDB e índices
# O banco é complementar e usa coleções próprias; nenhuma tabela relacional é alterada.
from pathlib import Path
import sys

(APP_HOME / "database.py").write_text(r'''import base64
import hashlib
import hmac
import os
import secrets
from functools import lru_cache
from pymongo import MongoClient, ASCENDING, DESCENDING

@lru_cache(maxsize=1)
def get_client():
    uri = os.environ.get("MONGO_URI")
    if not uri:
        raise RuntimeError("MONGO_URI não configurada.")
    return MongoClient(uri, appname="SaudeInfantilComplementar", serverSelectionTimeoutMS=8000, connectTimeoutMS=8000)

def get_db():
    name = os.environ.get("MONGO_DATABASE", "inteligencia_saude_infantil")
    return get_client()[name]

def ensure_indexes(db=None):
    db = db or get_db()
    db.eventos_integracao.create_index([("source_system", ASCENDING), ("event_id", ASCENDING)], unique=True)
    db.eventos_integracao.create_index([("child_external_id", ASCENDING), ("occurred_at", DESCENDING)])
    db.recebimentos_integracao.create_index([("event_id", ASCENDING), ("received_at", DESCENDING)])
    db.criancas.create_index([("source_system", ASCENDING), ("external_id", ASCENDING)], unique=True)
    db.criancas.create_index([("unit_external_id", ASCENDING), ("updated_at", DESCENDING)])
    db.monitoramentos.create_index([("source_system", ASCENDING), ("child_external_id", ASCENDING), ("rule_id", ASCENDING), ("occurrence_key", ASCENDING)], unique=True)
    db.monitoramentos.create_index([("status", ASCENDING), ("priority", ASCENDING), ("updated_at", DESCENDING)])
    db.resumos_alta.create_index([("document_id", ASCENDING)], unique=True)
    db.resumos_alta.create_index([("source_system", ASCENDING), ("child_external_id", ASCENDING), ("generated_at", DESCENDING)])
    db.auditoria.create_index([("entity_id", ASCENDING), ("created_at", DESCENDING)])
    db.auditoria.create_index([("actor", ASCENDING), ("created_at", DESCENDING)])
    db.usuarios.create_index([("username", ASCENDING)], unique=True)
    db.regras_monitoramento.create_index([("rule_id", ASCENDING)], unique=True)

def ping_and_prepare():
    db = get_db()
    db.command("ping")
    ensure_indexes(db)
    return db

def hash_password(password, salt=None):
    salt = salt or secrets.token_bytes(16)
    digest = hashlib.pbkdf2_hmac("sha256", password.encode(), salt, 310_000)
    return {"salt": base64.b64encode(salt).decode(), "digest": base64.b64encode(digest).decode()}

def verify_password(password, encoded):
    try:
        salt = base64.b64decode(encoded["salt"])
        expected = base64.b64decode(encoded["digest"])
    except (KeyError, ValueError, TypeError):
        return False
    actual = hashlib.pbkdf2_hmac("sha256", password.encode(), salt, 310_000)
    return hmac.compare_digest(actual, expected)

def audit(db, actor, action, entity_type, entity_id, details=None):
    from datetime import datetime, timezone
    db.auditoria.insert_one({
        "actor": actor, "action": action, "entity_type": entity_type,
        "entity_id": str(entity_id), "details": details or {},
        "created_at": datetime.now(timezone.utc),
    })
''', encoding="utf-8")
(APP_HOME / "requirements.txt").write_text("""pymongo>=4.6,<5\nfastapi>=0.110,<1\nuvicorn[standard]>=0.27,<1\nstreamlit>=1.32,<2\npandas>=2.1,<4\nplotly>=5.18,<8\npydantic>=2.6,<3\nreportlab>=4.1,<6\nrequests>=2.31,<3\nhttpx>=0.27,<1\npytest>=8,<10\nmongomock>=4.1,<5\n""", encoding="utf-8")
(APP_HOME / "CONTRATO_INTEGRACAO.md").write_text("""# Contrato de integração v1.0\n\nEnvie JSON por HTTPS com o cabeçalho `X-API-Key`. Campos obrigatórios: `event_id`, `source_system`, `schema_version` (`1.0`), `event_type`, `entity_type`, `entity_external_id`, `occurred_at` ISO-8601 com fuso e `data` objeto. `child_external_id` e `unit_external_id` são recomendados para projeção da criança e filtros; são opcionais em eventos sem esse vínculo.\n\nTipos: `gestacao`, `nascimento`, `internacao_neonatal`, `alta_maternidade`, `triagem`, `consulta`, `vacinacao`, `agendamento`. O adaptador aceita aliases documentados em `services.py` (`FIELD_ALIASES`); adapte-os ao contrato real sem presumir nomes de colunas do banco relacional. `data` não deve conter atributos alheios à finalidade.\n\n`POST /api/v1/integracao/registros` recebe um evento; `POST /api/v1/integracao/registros/lote` recebe `{\"events\": [...]}` (máximo 500); `GET /api/v1/integracao/registros/{event_id}?source_system=...` consulta o histórico; `GET /health` verifica a API e o MongoDB. Um reenvio idêntico é idempotente. Mesmo identificador com conteúdo corrigido incrementa versão e mantém histórico; eventos anteriores não substituem projeções mais recentes. Respostas inválidas usam HTTP 422 e erros de chave usam HTTP 401.\n\nTriagens: aliases canônicos configurados em `monitoring.py`. Status conhecidos são realizado, pendente e não realizado; valores desconhecidos permanecem como status não mapeado. Evento ausente é `SEM_INFORMACAO`, não confirmação de não realização.\n""", encoding="utf-8")
(APP_HOME / "README.md").write_text("""# Plataforma de Inteligência da Saúde Infantil\n\nAplicação complementar: o sistema relacional permanece como fonte oficial e envia eventos HTTP/JSON. Esta aplicação não acessa SQL Server.\n\nNo Colab, execute as células em ordem, informe as credenciais na célula 3 e use somente dados sintéticos. API FastAPI: `/docs`; interface Streamlit é iniciada pela célula 10; a célula 11 publica túneis Cloudflare efêmeros para demonstração. Nunca use dados clínicos reais nos túneis/Colab.\n\nArquivos auxiliares e o contrato são gerados nesta pasta durante as células 4–9. Instalação local: `pip install -r requirements.txt`. Credenciais são variáveis de ambiente em memória, nunca conteúdo de arquivos. Veja `CONTRATO_INTEGRACAO.md` para o schema, endpoints, idempotência e aliases adaptáveis.\n\nA plataforma guarda projeções mínimas, eventos recebidos, monitoramentos, resumos e auditoria. Prazos temporais começam desligados e exigem configuração/validação local. Ausência de evento é exibida como sem informação. Prioridades são operacionais e não substituem avaliação clínica. Protótipo para desenvolvimento; sujeito a validação de segurança, LGPD e governança antes de qualquer uso real.\n""", encoding="utf-8")
if str(APP_HOME) not in sys.path:
    sys.path.insert(0, str(APP_HOME))
db = __import__("database").ping_and_prepare()
print(f"MongoDB conectado; banco complementar preparado: {db.name}")
print("README.md, requirements.txt e CONTRATO_INTEGRACAO.md gerados sem credenciais.")

In [ ]:
# Célula 5 — Contratos e validação Pydantic
(APP_HOME / "models.py").write_text(r'''from datetime import datetime, timezone
from enum import Enum
from typing import Any
from pydantic import BaseModel, ConfigDict, Field, field_validator

class EventType(str, Enum):
    GESTACAO = "gestacao"
    NASCIMENTO = "nascimento"
    INTERNACAO_NEONATAL = "internacao_neonatal"
    ALTA_MATERNIDADE = "alta_maternidade"
    TRIAGEM = "triagem"
    CONSULTA = "consulta"
    VACINACAO = "vacinacao"
    AGENDAMENTO = "agendamento"

class IntegrationEvent(BaseModel):
    model_config = ConfigDict(extra="forbid")
    event_id: str = Field(min_length=1, max_length=160)
    source_system: str = Field(min_length=1, max_length=100)
    schema_version: str = Field(pattern=r"^\d+\.\d+$", max_length=20)
    event_type: EventType
    entity_type: str = Field(min_length=1, max_length=80)
    entity_external_id: str = Field(min_length=1, max_length=180)
    child_external_id: str | None = Field(default=None, max_length=180)
    unit_external_id: str | None = Field(default=None, max_length=180)
    occurred_at: datetime
    data: dict[str, Any]

    @field_validator("event_id", "source_system", "entity_external_id", "child_external_id", "unit_external_id")
    @classmethod
    def strip_identifiers(cls, value):
        if value is None:
            return value
        value = value.strip()
        if not value:
            raise ValueError("Identificadores não podem ser vazios.")
        return value

    @field_validator("occurred_at")
    @classmethod
    def require_timezone(cls, value):
        if value.tzinfo is None:
            raise ValueError("occurred_at deve incluir fuso horário.")
        return value.astimezone(timezone.utc)

class IntegrationBatch(BaseModel):
    events: list[IntegrationEvent] = Field(min_length=1, max_length=500)

class MonitorUpdate(BaseModel):
    status: str = Field(pattern=r"^(NOVO|EM_ACOMPANHAMENTO|PENDENTE|RESOLVIDO|DESCARTADO)$")
    note: str | None = Field(default=None, max_length=2000)
    resolution_reason: str | None = Field(default=None, max_length=1000)
    responsible: str | None = Field(default=None, max_length=160)

class SummaryValidation(BaseModel):
    professional: str = Field(min_length=1, max_length=160)
    note: str | None = Field(default=None, max_length=1000)
''', encoding="utf-8")
from importlib import invalidate_caches
invalidate_caches()
print("Modelos de eventos, lotes e atualização operacional gerados.")

In [ ]:
# Célula 6 — Serviço de recebimento, normalização e projeção mínima
# O adaptador aceita aliases configuráveis e mantém o payload original somente no registro do evento.
(APP_HOME / "services.py").write_text(r'''import hashlib
import json
from datetime import datetime, timezone
from uuid import uuid4
from bson import ObjectId
from database import audit

FIELD_ALIASES = {
    "nome": ("nome", "name", "child_name"),
    "data_nascimento": ("data_nascimento", "birth_date"),
    "sexo": ("sexo", "sex"),
    "peso_nascimento": ("peso_nascimento", "peso_ao_nascer", "birth_weight"),
    "comprimento_nascimento": ("comprimento_nascimento", "comprimento_ao_nascer", "birth_length"),
    "perimetro_cefalico": ("perimetro_cefalico", "perimetro_cefalico_nascimento", "head_circumference"),
    "data_alta": ("data_alta", "discharge_date"),
    "data_internacao": ("data_internacao", "admission_date"),
    "motivo_internacao": ("motivo_internacao", "admission_reason"),
    "tipo_triagem": ("tipo_triagem", "triagem", "screening_type"),
    "status": ("status", "situacao", "state"),
    "resultado": ("resultado", "result"),
    "data_procedimento": ("data_procedimento", "data_realizacao", "performed_at"),
    "vacina": ("vacina", "vaccine"),
    "dose": ("dose", "dose_name"),
    "data_aplicacao": ("data_aplicacao", "application_date"),
    "data_consulta": ("data_consulta", "appointment_date"),
    "data_agendamento": ("data_agendamento", "data_hora", "scheduled_at"),
    "alimentacao": ("alimentacao", "aleitamento", "feeding"),
    "orientacoes": ("orientacoes", "discharge_instructions"),
    "medicamentos": ("medicamentos", "medications"),
    "encaminhamentos": ("encaminhamentos", "referrals"),
    "diagnosticos_registrados": ("diagnosticos_registrados", "diagnoses_recorded"),
    "unidade_hospitalar": ("unidade_hospitalar", "hospital_name"),
    "unidade_referencia": ("unidade_referencia", "reference_unit"),
    "profissional": ("profissional", "professional"),
}

def utcnow():
    return datetime.now(timezone.utc)

def as_utc(value):
    if isinstance(value, str):
        value = datetime.fromisoformat(value.replace("Z", "+00:00"))
    if value.tzinfo is None:
        value = value.replace(tzinfo=timezone.utc)
    return value.astimezone(timezone.utc)

def normalized_projection(data):
    result = {}
    for target, aliases in FIELD_ALIASES.items():
        for alias in aliases:
            if alias in data:
                result[target] = data[alias]
                break
    return result

def event_hash(event):
    encoded = json.dumps(event, sort_keys=True, ensure_ascii=False, default=str, separators=(",", ":")).encode()
    return hashlib.sha256(encoded).hexdigest()

def json_safe(value):
    if isinstance(value, ObjectId):
        return str(value)
    if isinstance(value, datetime):
        return value.replace(tzinfo=value.tzinfo or timezone.utc).astimezone(timezone.utc).isoformat()
    if isinstance(value, list):
        return [json_safe(item) for item in value]
    if isinstance(value, dict):
        return {key: json_safe(item) for key, item in value.items() if key != "_id"}
    return value

def process_event(db, event):
    event = event.model_dump(mode="python") if hasattr(event, "model_dump") else dict(event)
    if len(json.dumps(event, default=str).encode()) > 256_000:
        raise ValueError("Evento excede o limite de 256 KB.")
    now = utcnow()
    key = {"source_system": event["source_system"], "event_id": event["event_id"]}
    raw_type = event["event_type"]
    event_type = raw_type.value if hasattr(raw_type, "value") else str(raw_type)
    digest = event_hash(event)
    previous = db.eventos_integracao.find_one(key)
    if previous and previous.get("payload_hash") == digest:
        db.recebimentos_integracao.insert_one({**key, "status": "DUPLICADO", "received_at": now})
        return {"event_id": event["event_id"], "status": "DUPLICADO", "duplicate": True}

    history = list(previous.get("history", [])) if previous else []
    version = int(previous.get("version", 0)) + 1 if previous else 1
    receipt_status = "CORRIGIDO" if previous else "PROCESSADO"
    if previous:
        history.append({"version": previous["version"], "payload_hash": previous.get("payload_hash"), "received_at": previous.get("updated_at")})
    occurred = as_utc(event["occurred_at"])
    event_document = {
        **event, **key, "event_type": event_type, "occurred_at": occurred, "payload_hash": digest,
        "version": version, "history": history, "received_at": previous.get("received_at", now) if previous else now,
        "updated_at": now, "processing_status": "RECEBIDO",
    }
    db.eventos_integracao.replace_one(key, event_document, upsert=True)

    child_id = event.get("child_external_id")
    projection_status = "SEM_PROJECAO"
    if child_id:
        from monitoring import canonical_screening
        child_key = {"source_system": event["source_system"], "external_id": child_id}
        child = db.criancas.find_one(child_key) or {}
        normalized = normalized_projection(event.get("data", {}))
        projection_key = event_type
        if event_type == "triagem":
            triage_key = canonical_screening(normalized.get("tipo_triagem")) or str(normalized.get("tipo_triagem") or event["entity_external_id"]).casefold()
            projection_key = f"triagem:{triage_key}"
        current = (child.get("projections") or {}).get(projection_key)
        current_time = as_utc(current["occurred_at"]) if current else occurred
        if current is None or occurred >= current_time:
            merged_data = {**(current or {}).get("data", {}), **normalized}
            projection = {
                "event_id": event["event_id"], "entity_external_id": event["entity_external_id"],
                "event_type": event_type, "occurred_at": occurred,
                "unit_external_id": event.get("unit_external_id"), "data": merged_data,
            }
            update = {
                "source_system": event["source_system"], "external_id": child_id,
                "unit_external_id": event.get("unit_external_id") or child.get("unit_external_id"),
                "updated_at": now, f"projections.{projection_key}": projection,
            }
            if event_type == "nascimento":
                update["birth_event_received"] = True
            db.criancas.update_one(child_key, {"$setOnInsert": {"created_at": now}, "$set": update}, upsert=True)
            projection_status = "PROJETADO"
        else:
            projection_status = "FORA_DE_ORDEM"

    if str(event.get("entity_type", "")).casefold() in ("gestacao", "pregnancy"):
        pregnancy_key = {"source_system": event["source_system"], "external_id": event["entity_external_id"]}
        db.gestacoes.update_one(pregnancy_key, {"$set": {
            **pregnancy_key, "child_external_id": child_id,
            "occurred_at": occurred, "data": normalized_projection(event.get("data", {})),
            "updated_at": now,
        }, "$setOnInsert": {"created_at": now}}, upsert=True)

    db.eventos_integracao.update_one(key, {"$set": {"processing_status": projection_status}})
    db.recebimentos_integracao.insert_one({**key, "status": receipt_status, "projection_status": projection_status, "received_at": now})
    from monitoring import refresh_child_monitorings
    if child_id:
        refresh_child_monitorings(db, event["source_system"], child_id, actor="integracao")
        db.resumos_alta.update_many({
            "source_system": event["source_system"], "child_external_id": child_id,
            "status": {"$in": ["RASCUNHO", "PENDENTE_VALIDACAO", "VALIDADO"]},
        }, {"$set": {"status": "ATUALIZADO", "source_update_received_at": now, "updated_at": now}})
    return {"event_id": event["event_id"], "status": receipt_status, "projection_status": projection_status, "version": version}

def get_event(db, event_id, source_system=None):
    query = {"event_id": event_id}
    if source_system:
        query["source_system"] = source_system
    return json_safe(db.eventos_integracao.find_one(query))

def children(db, filters=None):
    filters = filters or {}
    query = {}
    if filters.get("unit_external_id"):
        query["unit_external_id"] = filters["unit_external_id"]
    if filters.get("external_id"):
        query["external_id"] = {"$regex": filters["external_id"], "$options": "i"}
    return [json_safe(item) for item in db.criancas.find(query).sort("updated_at", -1)]

def create_discharge_summary(db, source_system, child_external_id, actor):
    child = db.criancas.find_one({"source_system": source_system, "external_id": child_external_id})
    if not child:
        raise LookupError("Criança não encontrada na projeção recebida.")
    events = list(db.eventos_integracao.find({"source_system": source_system, "child_external_id": child_external_id}).sort("occurred_at", 1))
    latest = {}
    repeated = {"consulta", "vacinacao", "agendamento", "triagem"}
    for item in events:
        kind = item.get("event_type")
        normalized = normalized_projection(item.get("data", {}))
        if kind in repeated:
            latest.setdefault(kind, []).append({"event_id": item.get("event_id"), "occurred_at": item.get("occurred_at"), **normalized})
        else:
            latest[kind] = normalized
    triage_events = latest.pop("triagem", [])
    from monitoring import SCREENINGS, canonical_screening
    triage_by_type = {}
    for item in triage_events:
        key = canonical_screening(item.get("tipo_triagem"))
        if key:
            triage_by_type[key] = item
    latest["triagens"] = [{
        "tipo": key, "nome": item["label"],
        "status_origem": triage_by_type.get(key, {}).get("status", "SEM_INFORMACAO"),
        "data": triage_by_type.get(key, {}).get("data_procedimento"),
        "resultado_origem": triage_by_type.get(key, {}).get("resultado"),
    } for key, item in SCREENINGS.items()]

    birth = latest.get("nascimento", {})
    discharge = latest.get("alta_maternidade", {})
    identity = {"nome": birth.get("nome"), "data_nascimento": birth.get("data_nascimento"),
                "sexo": birth.get("sexo"), "unidade_referencia": child.get("unit_external_id")}
    missing = [label for key, label in (("nome", "identificação da criança"), ("data_nascimento", "data de nascimento"),
        ("sexo", "sexo registrado"), ("unidade_hospitalar", "unidade hospitalar"), ("data_alta", "data da alta"))
        if not (identity if key in identity else {**birth, **discharge}).get(key)]
    if not child.get("unit_external_id"):
        missing.append("unidade de referência")
    sections = {"identificacao": identity, **latest}
    summary_id = str(uuid4())
    summary = {
        "document_id": summary_id, "source_system": source_system,
        "child_external_id": child_external_id, "unit_external_id": child.get("unit_external_id"),
        "status": "RASCUNHO", "generated_at": utcnow(), "generated_by": actor,
        "source_event_ids": [item.get("event_id") for item in events],
        "sections": sections, "missing_fields": sorted(set(missing)),
    }
    db.resumos_alta.insert_one(summary)
    audit(db, actor, "resumo_gerado", "resumo_alta", summary_id, {"child_external_id": child_external_id})
    return json_safe(summary)

def update_monitoring(db, monitor_id, update_data, actor):
    if update_data["status"] == "RESOLVIDO" and not (update_data.get("resolution_reason") or "").strip():
        raise ValueError("Informe a justificativa para resolver a ocorrência.")
    if update_data["status"] == "DESCARTADO" and not (update_data.get("resolution_reason") or "").strip():
        raise ValueError("Informe a justificativa para descartar a ocorrência.")
    item = db.monitoramentos.find_one({"_id": ObjectId(monitor_id)})
    if not item:
        raise LookupError("Ocorrência não encontrada.")
    now = utcnow()
    change = {"status": update_data["status"], "updated_at": now}
    for field in ("note", "responsible"):
        if update_data.get(field) is not None:
            change[field] = update_data[field]
    if update_data["status"] in ("RESOLVIDO", "DESCARTADO"):
        change.update({"resolution_reason": update_data.get("resolution_reason"), "resolved_at": now})
    history_item = {"actor": actor, "from": item.get("status"), "to": update_data["status"],
                    "note": update_data.get("note"), "at": now}
    db.monitoramentos.update_one({"_id": item["_id"]}, {"$set": change, "$push": {"history": history_item}})
    audit(db, actor, "monitoramento_atualizado", "monitoramento", monitor_id, history_item)
    return json_safe(db.monitoramentos.find_one({"_id": item["_id"]}))
''', encoding="utf-8")
print("Serviços de ingestão, versões, projeções mínimas e resumos com histórico gerados.")

In [ ]:
# Célula 7 — Motor de monitoramento configurável
# Regras padrão são operacionais; não classificam risco clínico e não definem prazos clínicos.
(APP_HOME / "monitoring.py").write_text(r'''import unicodedata
from datetime import datetime, timedelta, timezone

SCREENINGS = {
    "pezinho": {"label": "Teste do pezinho", "aliases": {"pezinho", "teste_pezinho", "teste do pezinho", "newborn_blood_spot"}},
    "orelhinha": {"label": "Teste da orelhinha", "aliases": {"orelhinha", "teste_orelhinha", "teste da orelhinha", "hearing"}},
    "olhinho": {"label": "Teste do olhinho", "aliases": {"olhinho", "teste_olhinho", "teste do olhinho", "eye"}},
    "coracaozinho": {"label": "Teste do coraçãozinho", "aliases": {"coracaozinho", "coraçãozinho", "teste_coracaozinho", "teste do coraçãozinho", "pulse_oximetry"}},
}
PRIORITIES = {"BAIXA", "MEDIA", "ALTA", "CRITICA"}
DONE_STATES = {"realizado", "realizada", "concluido", "concluida", "done", "completed"}
PENDING_STATES = {"pendente", "pending", "agendado", "agendada"}
NOT_DONE_STATES = {"nao_realizado", "nao_realizada", "not_performed", "not_done"}
TEMPORAL_RULES = {
    "alta_sem_consulta": ("ALTA_SEM_CONSULTA", "Sem evento de consulta após o intervalo configurado para acompanhamento da alta."),
    "dados_desatualizados": ("DADOS_DESATUALIZADOS", "Nenhum evento de origem recebido dentro do intervalo configurado."),
}

def ensure_default_rules(db):
    now = datetime.now(timezone.utc)
    for key, screening in SCREENINGS.items():
        db.regras_monitoramento.update_one({"rule_id": f"triagem_sem_informacao_{key}"}, {"$setOnInsert": {
            "rule_id": f"triagem_sem_informacao_{key}", "enabled": True,
            "priority": "BAIXA", "screening_key": key,
            "description": f"Sem evento de origem recebido para {screening['label']}; solicitar verificação da informação.",
            "clinical_protocol": False, "created_at": now,
        }}, upsert=True)
    for rule_id, description in [
        ("alta_sem_consulta", "Acompanhamento após alta; configure o prazo local aprovado antes de ativar."),
        ("dados_desatualizados", "Atualização de dados; configure o intervalo local aprovado antes de ativar."),
    ]:
        db.regras_monitoramento.update_one({"rule_id": rule_id}, {"$setOnInsert": {
            "rule_id": rule_id, "enabled": False, "priority": "BAIXA", "threshold_days": None,
            "description": description, "clinical_protocol": False, "created_at": now,
        }}, upsert=True)

def canonical_screening(value):
    text = str(value or "").strip().casefold()
    for key, item in SCREENINGS.items():
        if text in {alias.casefold() for alias in item["aliases"]}:
            return key
    return None

def canonical_status(value):
    text = unicodedata.normalize("NFD", str(value or "").casefold())
    text = "".join(char for char in text if unicodedata.category(char) != "Mn")
    return "_".join(text.replace("-", " ").split())

def as_utc(value):
    if isinstance(value, str):
        value = datetime.fromisoformat(value.replace("Z", "+00:00"))
    if value.tzinfo is None:
        value = value.replace(tzinfo=timezone.utc)
    return value.astimezone(timezone.utc)

def resolve_occurrence(db, key, actor, now, reason):
    existing = db.monitoramentos.find_one(key)
    if existing and existing.get("status") not in ("RESOLVIDO", "DESCARTADO"):
        db.monitoramentos.update_one({"_id": existing["_id"]}, {
            "$set": {"status": "RESOLVIDO", "updated_at": now, "resolution_reason": reason, "resolved_at": now},
            "$push": {"history": {"actor": actor, "from": existing.get("status"), "to": "RESOLVIDO", "at": now}},
        })

def record_occurrence(db, key, fields, actor, now):
    result = db.monitoramentos.update_one(key, {"$setOnInsert": {
        **key, **fields, "status": fields.get("initial_status", "NOVO"),
        "identified_at": now, "created_at": now, "updated_at": now,
        "responsible": None, "note": None,
        "history": [{"actor": actor, "from": None, "to": fields.get("initial_status", "NOVO"), "at": now}],
    }}, upsert=True)
    return str(result.upserted_id) if result.upserted_id else None

def refresh_temporal_rules(db, child, source_system, child_external_id, actor, now):
    changed = []
    for rule_id, (occurrence_type, default_description) in TEMPORAL_RULES.items():
        rule = db.regras_monitoramento.find_one({"rule_id": rule_id}) or {}
        threshold = rule.get("threshold_days")
        if not rule.get("enabled") or not isinstance(threshold, int) or threshold < 1:
            continue
        key = {"source_system": source_system, "child_external_id": child_external_id,
               "rule_id": rule_id, "occurrence_key": rule_id}
        trigger = False
        reason = rule.get("description", default_description)
        if rule_id == "alta_sem_consulta":
            discharge = child.get("projections", {}).get("alta_maternidade")
            if discharge and discharge.get("occurred_at"):
                discharge_time = as_utc(discharge["occurred_at"])
                due = now >= discharge_time + timedelta(days=threshold)
                consultations = list(db.eventos_integracao.find({
                    "source_system": source_system, "child_external_id": child_external_id,
                    "event_type": "consulta",
                }))
                follow_up = any(as_utc(item["occurred_at"]) > discharge_time for item in consultations)
                trigger = due and not follow_up
                reason = f"Nenhum evento de consulta foi recebido em {threshold} dias configurados após a alta."
        else:
            last_update = child.get("updated_at")
            if last_update:
                trigger = now >= as_utc(last_update) + timedelta(days=threshold)
                reason = f"Nenhum evento foi recebido em {threshold} dias configurados."

        if trigger:
            created = record_occurrence(db, key, {
                "unit_external_id": child.get("unit_external_id"), "type": occurrence_type,
                "description": reason, "rule_reason": rule.get("description", default_description),
                "data_state": "SEM_EVENTO_NO_INTERVALO_CONFIGURADO", "priority": rule.get("priority", "BAIXA"),
                "threshold_days": threshold, "initial_status": "PENDENTE",
            }, actor, now)
            if created:
                changed.append(created)
        else:
            resolve_occurrence(db, key, actor, now, "Evento posterior recebido ou intervalo configurado ainda não transcorrido.")
    return changed

def refresh_child_monitorings(db, source_system, child_external_id, actor="motor_regras"):
    ensure_default_rules(db)
    child = db.criancas.find_one({"source_system": source_system, "external_id": child_external_id})
    if not child:
        return []
    projections = child.get("projections", {})
    if not projections.get("nascimento") and not projections.get("alta_maternidade"):
        return []
    events = list(db.eventos_integracao.find({
        "source_system": source_system, "child_external_id": child_external_id, "event_type": "triagem",
    }).sort("occurred_at", -1))
    latest = {}
    for event in events:
        data = event.get("data", {})
        key = canonical_screening(data.get("tipo_triagem") or data.get("triagem") or data.get("screening_type"))
        if key and key not in latest:
            latest[key] = event

    changed = []
    now = datetime.now(timezone.utc)
    for key, screening in SCREENINGS.items():
        rule = db.regras_monitoramento.find_one({"rule_id": f"triagem_sem_informacao_{key}"}) or {}
        if not rule.get("enabled", False):
            continue
        event = latest.get(key)
        base_key = {"source_system": source_system, "child_external_id": child_external_id, "rule_id": rule["rule_id"]}
        missing_key = {**base_key, "occurrence_key": key}
        if not event:
            description = f"Sem informação recebida sobre {screening['label']}. Verifique com a origem; ausência de evento não confirma não realização."
            created = record_occurrence(db, missing_key, {
                "unit_external_id": child.get("unit_external_id"), "type": "TRIAGEM_SEM_INFORMACAO",
                "description": description, "rule_reason": rule.get("description", description),
                "data_state": "SEM_INFORMACAO", "priority": rule.get("priority", "BAIXA"), "initial_status": "NOVO",
            }, actor, now)
            if created:
                changed.append(created)
            continue

        raw_status = event.get("data", {}).get("status") or event.get("data", {}).get("situacao") or event.get("data", {}).get("state")
        state = canonical_status(raw_status)
        resolve_occurrence(db, missing_key, actor, now, "Evento de triagem recebido do sistema de origem.")
        if state in DONE_STATES:
            for suffix in ("pendente", "nao_realizado", "status_nao_mapeado"):
                resolve_occurrence(db, {**base_key, "occurrence_key": f"{key}:{suffix}"}, actor, now,
                    "A origem atualizou a triagem como realizada.")
            continue

        if state in PENDING_STATES:
            occurrence_type, suffix = "TRIAGEM_PENDENTE", "pendente"
            data_state, initial_status = "PENDENTE_INFORMADO_PELA_ORIGEM", "PENDENTE"
            description = f"A origem informou situação pendente para {screening['label']}. Confirme a atualização com a equipe responsável."
        elif state in NOT_DONE_STATES:
            occurrence_type, suffix = "TRIAGEM_NAO_REALIZADA", "nao_realizado"
            data_state, initial_status = "NAO_REALIZADO_CONFIRMADO_PELA_ORIGEM", "NOVO"
            description = f"A origem confirmou que {screening['label']} não foi realizado. Avaliação e encaminhamento cabem à equipe responsável."
        else:
            occurrence_type, suffix = "TRIAGEM_STATUS_NAO_MAPEADO", "status_nao_mapeado"
            data_state, initial_status = "STATUS_ORIGEM_NAO_INTERPRETADO", "PENDENTE"
            description = f"Há um registro de {screening['label']}, mas o status de origem não foi reconhecido. Confirme o mapeamento."
        occurrence_key = f"{key}:{suffix}"
        created = record_occurrence(db, {**base_key, "occurrence_key": occurrence_key}, {
            "unit_external_id": child.get("unit_external_id"), "type": occurrence_type,
            "description": description, "rule_reason": f"Status recebido da origem: {raw_status or 'sem status'}.",
            "data_state": data_state, "source_status": raw_status,
            "priority": rule.get("priority", "BAIXA"), "initial_status": initial_status,
        }, actor, now)
        if created:
            changed.append(created)

    changed.extend(refresh_temporal_rules(db, child, source_system, child_external_id, actor, now))
    return changed

def configure_rule(db, rule_id, enabled, priority, threshold_days=None):
    if priority not in PRIORITIES:
        raise ValueError("Prioridade operacional inválida.")
    if threshold_days is not None and (not isinstance(threshold_days, int) or threshold_days < 1):
        raise ValueError("O intervalo deve ser inteiro positivo ou vazio.")
    rule = db.regras_monitoramento.find_one({"rule_id": rule_id})
    if not rule:
        raise LookupError("Regra não encontrada.")
    if rule_id in TEMPORAL_RULES and enabled and threshold_days is None:
        raise ValueError("Defina e valide o intervalo local antes de ativar uma regra temporal.")
    db.regras_monitoramento.update_one({"rule_id": rule_id}, {"$set": {
        "enabled": bool(enabled), "priority": priority, "threshold_days": threshold_days,
        "updated_at": datetime.now(timezone.utc),
    }})
    return db.regras_monitoramento.find_one({"rule_id": rule_id})
''', encoding="utf-8")
print("Regras configuráveis geradas; regras temporais começam desativadas.")

In [ ]:
# Célula 8 — API FastAPI para integração
(APP_HOME / "api.py").write_text(r'''import asyncio
import hmac
import os
from contextlib import asynccontextmanager
from datetime import datetime, timezone
from fastapi import Depends, FastAPI, Header, HTTPException, Request
from fastapi.exceptions import RequestValidationError
from fastapi.responses import JSONResponse
from database import get_db, hash_password, ping_and_prepare
from models import IntegrationBatch, IntegrationEvent
from services import process_event, get_event
from monitoring import ensure_default_rules, refresh_child_monitorings

def recheck_all_monitorings():
    db = get_db()
    for child in db.criancas.find({}, {"source_system": 1, "external_id": 1}):
        if child.get("source_system") and child.get("external_id"):
            refresh_child_monitorings(db, child["source_system"], child["external_id"], actor="motor_regras")

async def monitoring_loop():
    interval = max(60, int(os.environ.get("MONITOR_RECHECK_SECONDS", "3600")))
    while True:
        try:
            await asyncio.to_thread(recheck_all_monitorings)
        except Exception as exc:
            print(f"[monitoramento] Reavaliação falhou: {type(exc).__name__}")
        await asyncio.sleep(interval)

@asynccontextmanager
async def lifespan(app):
    db = ping_and_prepare()
    ensure_default_rules(db)
    username = os.environ.get("APP_ADMIN_USERNAME")
    password = os.environ.get("APP_ADMIN_PASSWORD")
    if username and password:
        db.usuarios.update_one({"username": username}, {"$setOnInsert": {
            "username": username, "password": hash_password(password), "role": "ADMIN",
            "active": True, "created_at": datetime.now(timezone.utc),
        }}, upsert=True)
    try:
        recheck_all_monitorings()
    except Exception as exc:
        print(f"[monitoramento] Avaliação inicial falhou: {type(exc).__name__}")
    task = asyncio.create_task(monitoring_loop())
    try:
        yield
    finally:
        task.cancel()
        try:
            await task
        except asyncio.CancelledError:
            pass

app = FastAPI(title="Integração de Saúde Infantil", version="1.0.0",
              description="Recebe eventos do sistema relacional sem conexão direta ao SQL Server.", lifespan=lifespan)

def integration_auth(x_api_key: str | None = Header(default=None, alias="X-API-Key")):
    expected = os.environ.get("API_INTEGRATION_KEY", "")
    if not expected or not x_api_key or not hmac.compare_digest(x_api_key, expected):
        raise HTTPException(status_code=401, detail={"code": "unauthorized", "message": "Chave de integração inválida."})
    return "sistema_relacional"

@app.exception_handler(RequestValidationError)
async def validation_error_handler(request: Request, exc: RequestValidationError):
    try:
        get_db().recebimentos_integracao.insert_one({
            "status": "REJEITADO_VALIDACAO", "path": request.url.path,
            "error_codes": [item.get("type", "invalid") for item in exc.errors()],
            "received_at": datetime.now(timezone.utc),
        })
    except Exception:
        pass
    return JSONResponse(status_code=422, content={"ok": False, "error": {
        "code": "validation_error", "message": "O registro não corresponde ao contrato de integração.",
        "fields": [{"field": ".".join(str(part) for part in item.get("loc", [])), "message": item.get("msg", "Inválido.")} for item in exc.errors()],
    }})

@app.get("/health")
def health():
    try:
        db = get_db()
        db.command("ping")
        return {"ok": True, "service": "health-infantil-integration", "database": db.name}
    except Exception:
        raise HTTPException(status_code=503, detail={"code": "database_unavailable", "message": "Banco complementar indisponível."})

@app.post("/api/v1/integracao/registros", status_code=202)
def receive_event(event: IntegrationEvent, actor=Depends(integration_auth)):
    try:
        return {"ok": True, "data": process_event(get_db(), event)}
    except ValueError as exc:
        raise HTTPException(status_code=422, detail={"code": "invalid_event", "message": str(exc)})

@app.post("/api/v1/integracao/registros/lote", status_code=202)
def receive_batch(batch: IntegrationBatch, actor=Depends(integration_auth)):
    results = []
    for event in batch.events:
        try:
            results.append({"ok": True, "data": process_event(get_db(), event)})
        except Exception as exc:
            results.append({"ok": False, "event_id": event.event_id, "error": {"code": "processing_error", "message": type(exc).__name__}})
    return {"ok": all(item["ok"] for item in results), "received": len(results), "results": results}

@app.get("/api/v1/integracao/registros/{event_id}")
def read_event(event_id: str, source_system: str | None = None, actor=Depends(integration_auth)):
    event = get_event(get_db(), event_id, source_system)
    if not event:
        raise HTTPException(status_code=404, detail={"code": "not_found", "message": "Evento não encontrado."})
    return {"ok": True, "data": event}
''', encoding="utf-8")
print("api.py gerado com health, autenticação, integração e reavaliação periódica de regras.")

In [ ]:
# Célula 9 — Aplicação Streamlit integrada
(APP_HOME / "app.py").write_text(r'''import html
import json
import os
import sys
from datetime import datetime, timezone
from io import BytesIO
import pandas as pd
import plotly.express as px
import streamlit as st
sys.path.insert(0, os.environ["APP_HOME"])
from database import get_db, verify_password, hash_password, audit
from services import children, update_monitoring, create_discharge_summary, json_safe
from monitoring import SCREENINGS, TEMPORAL_RULES, configure_rule
from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle

st.set_page_config(page_title="Saúde Infantil | Monitoramento", page_icon="+", layout="wide")
PALETTE = ["#147d77", "#e5a545", "#c85b4a", "#427c9b", "#6e9a6d"]

def login():
    st.title("Plataforma de Saúde Infantil")
    st.caption("Acesso restrito. Colab é ambiente de demonstração; use somente dados sintéticos.")
    with st.form("login"):
        username = st.text_input("Usuário")
        password = st.text_input("Senha", type="password")
        submit = st.form_submit_button("Entrar", type="primary")
    if submit:
        user = get_db().usuarios.find_one({"username": username, "active": True})
        if user and verify_password(password, user.get("password", {})):
            st.session_state["user"] = {"username": user["username"], "role": user.get("role", "LEITURA")}
            st.rerun()
        st.error("Usuário ou senha inválidos.")

if "user" not in st.session_state:
    login()
    st.stop()
user = st.session_state["user"]
db = get_db()
role = user["role"]
with st.sidebar:
    st.markdown("## SAÚDE INFANTIL")
    st.caption(f"{user['username']} · {role}")
    page = st.radio("Navegação", ["Visão geral", "Inteligência de dados", "Central de monitoramento", "Crianças", "Resumo digital de alta", "Integração", "Auditoria"])
    if st.button("Sair", use_container_width=True):
        st.session_state.pop("user", None)
        st.rerun()
all_children = children(db)
all_events = list(db.eventos_integracao.find().sort("occurred_at", -1))
all_monitors = list(db.monitoramentos.find().sort("updated_at", -1))

def child_name(child):
    return child.get("projections", {}).get("nascimento", {}).get("data", {}).get("nome") or child.get("external_id", "ID sem informação")

def birth_date(child):
    return child.get("projections", {}).get("nascimento", {}).get("data", {}).get("data_nascimento")

def age_days(child, today=None):
    value = birth_date(child)
    if not value:
        return None
    try:
        return ((today or datetime.now(timezone.utc).date()) - datetime.fromisoformat(str(value)[:10]).date()).days
    except ValueError:
        return None

def age_band(age):
    if age is None or age < 0:
        return "Sem informação"
    if age <= 7:
        return "0–7 dias"
    if age <= 28:
        return "8–28 dias"
    if age <= 365:
        return "29–365 dias"
    return "Mais de 1 ano"

def dashboard_metrics():
    newborns = sum(1 for child in all_children if age_days(child) is not None and 0 <= age_days(child) <= 28)
    open_items = [m for m in all_monitors if m.get("status") not in ("RESOLVIDO", "DESCARTADO")]
    complete_fields = 0
    for child in all_children:
        data = child.get("projections", {}).get("nascimento", {}).get("data", {})
        complete_fields += sum(bool(value) for value in (child.get("unit_external_id"), data.get("data_nascimento"), data.get("sexo"), data.get("peso_nascimento"), child.get("birth_event_received")))
    completeness = round(100 * complete_fields / (5 * len(all_children))) if all_children else 0
    return {
        "Crianças recebidas": len(all_children), "Recém-nascidos (0–28 dias)": newborns,
        "Acompanhamentos em aberto": len(open_items), "Prioridade alta/crítica": sum(m.get("priority") in ("ALTA", "CRITICA") for m in open_items),
        "Altas registradas": sum(1 for c in all_children if "alta_maternidade" in c.get("projections", {})),
        "Triagens sem informação": sum(1 for m in open_items if m.get("data_state") == "SEM_INFORMACAO"),
        "Agendamentos recebidos": sum(1 for e in all_events if e.get("event_type") == "agendamento"),
        "Vacinas pendentes informadas": sum(1 for e in all_events if e.get("event_type") == "vacinacao" and str(e.get("data", {}).get("status", "")).casefold() == "pendente"),
        "Completude campos-base": completeness,
    }

def render_dashboard():
    st.title("Visão geral")
    metrics = dashboard_metrics()
    columns = st.columns(3)
    for index, (label, value) in enumerate(metrics.items()):
        columns[index % 3].metric(label, f"{value}%" if label == "Completude campos-base" else value)
    if not all_children:
        st.info("Ainda não há registros. Integre a origem ou carregue a demonstração sintética na célula 12.")
    if all_events:
        frame = pd.DataFrame([{ "Tipo": e.get("event_type"), "Recebido": e.get("received_at"), "Unidade": e.get("unit_external_id") or "Sem informação" } for e in all_events])
        chart = frame.groupby("Tipo", dropna=False).size().reset_index(name="Eventos")
        st.plotly_chart(px.bar(chart, x="Tipo", y="Eventos", color="Tipo", color_discrete_sequence=PALETTE), use_container_width=True)

def render_intelligence():
    st.title("Inteligência de dados")
    if not all_events and not all_children:
        st.info("Os indicadores serão calculados conforme a origem enviar registros. Não há números de demonstração fixos.")
        return
    c1, c2, c3 = st.columns(3)
    since = c1.date_input("Período inicial", value=None)
    until = c2.date_input("Período final", value=None)
    units = sorted({e.get("unit_external_id") for e in all_events if e.get("unit_external_id")})
    unit = c3.selectbox("Unidade", ["Todas", *units])
    types = sorted({e.get("event_type") for e in all_events if e.get("event_type")})
    selected_types = st.multiselect("Tipo de evento", types, default=types)
    source_statuses = sorted({str(e.get("data", {}).get("status")) for e in all_events if e.get("data", {}).get("status")})
    selected_statuses = st.multiselect("Status recebido", source_statuses, default=source_statuses)
    filtered = []
    for event in all_events:
        occurred = event.get("occurred_at")
        event_status = str(event.get("data", {}).get("status", ""))
        if not occurred or event.get("event_type") not in selected_types:
            continue
        if unit != "Todas" and event.get("unit_external_id") != unit:
            continue
        day = occurred.date()
        if (since and day < since) or (until and day > until):
            continue
        if event_status and event_status not in selected_statuses:
            continue
        filtered.append(event)
    event_frame = pd.DataFrame([{ "Data": e["occurred_at"], "Tipo": e.get("event_type"), "Unidade": e.get("unit_external_id") or "Sem informação" } for e in filtered])
    if not event_frame.empty:
        event_frame["Data"] = pd.to_datetime(event_frame["Data"], utc=True).dt.date
        left, right = st.columns(2)
        with left:
            st.plotly_chart(px.histogram(event_frame, x="Tipo", color="Tipo", title="Eventos por tipo", color_discrete_sequence=PALETTE), use_container_width=True)
            unit_frame = pd.DataFrame([{ "Unidade": c.get("unit_external_id") or "Sem informação" } for c in all_children])
            if not unit_frame.empty:
                st.plotly_chart(px.histogram(unit_frame, x="Unidade", color="Unidade", title="Crianças por unidade", color_discrete_sequence=PALETTE), use_container_width=True)
        with right:
            timeline = event_frame.groupby("Data").size().reset_index(name="Eventos")
            st.plotly_chart(px.line(timeline, x="Data", y="Eventos", markers=True, title="Evolução de eventos"), use_container_width=True)
            monitor_frame = pd.DataFrame([{ "Status": m.get("status"), "Prioridade": m.get("priority"), "Unidade": m.get("unit_external_id") or "Sem informação" } for m in all_monitors])
            if not monitor_frame.empty:
                st.plotly_chart(px.histogram(monitor_frame, x="Status", color="Prioridade", title="Monitoramentos por status/prioridade", color_discrete_sequence=PALETTE), use_container_width=True)
        births = [e for e in filtered if e.get("event_type") == "nascimento"]
        discharges = [e for e in filtered if e.get("event_type") == "alta_maternidade"]
        triages = [e for e in filtered if e.get("event_type") == "triagem"]
        a, b, c = st.columns(3)
        for column, rows, title in ((a, births, "Nascimentos por período"), (b, discharges, "Altas por período"), (c, triages, "Triagens informadas")):
            if rows:
                frame = pd.DataFrame([{ "Data": e.get("occurred_at"), "Status": e.get("data", {}).get("status") or "Sem informação" } for e in rows])
                frame["Data"] = pd.to_datetime(frame["Data"], utc=True).dt.date
                column.plotly_chart(px.histogram(frame, x="Data", color="Status", title=title, color_discrete_sequence=PALETTE), use_container_width=True)
            else:
                column.caption(f"{title}: sem eventos recebidos.")
    child_units = sorted({c.get("unit_external_id") for c in all_children if c.get("unit_external_id")})
    age_unit = st.selectbox("Unidade no gráfico etário", ["Todas", *child_units])
    ages = [(child, age_band(age_days(child))) for child in all_children if age_unit == "Todas" or child.get("unit_external_id") == age_unit]
    bands = sorted({band for _, band in ages})
    selected_bands = st.multiselect("Faixa etária", bands, default=bands)
    age_rows = [{"Faixa etária": band, "Unidade": child.get("unit_external_id") or "Sem informação"} for child, band in ages if band in selected_bands]
    if age_rows:
        st.plotly_chart(px.histogram(pd.DataFrame(age_rows), x="Faixa etária", color="Unidade", title="Crianças por faixa etária e unidade", color_discrete_sequence=PALETTE), use_container_width=True)
    else:
        st.info("Sem crianças para as faixas etárias selecionadas.")
    st.subheader("Qualidade dos dados")
    issues = []
    for child in all_children:
        data = child.get("projections", {}).get("nascimento", {}).get("data", {})
        missing = [field for field in ("data_nascimento", "sexo", "peso_nascimento") if not data.get(field)]
        if not child.get("unit_external_id"):
            missing.append("unidade de referência")
        if missing:
            issues.append({"Criança": child_name(child), "ID de origem": child.get("external_id"), "Lacunas": ", ".join(missing)})
    if issues:
        frame = pd.DataFrame(issues)
        st.dataframe(frame, use_container_width=True, hide_index=True)
        st.download_button("Exportar inconsistências CSV", frame.to_csv(index=False).encode("utf-8-sig"), "qualidade_dos_dados.csv", "text/csv")
    else:
        st.success("Nenhuma lacuna nos campos-base avaliados.")
    st.caption("Ausência de evento é 'sem informação recebida'; não confirma que um procedimento não foi realizado.")

def render_monitoring():
    st.title("Central de monitoramento")
    priorities = sorted({m.get("priority", "BAIXA") for m in all_monitors}) or ["BAIXA"]
    statuses = sorted({m.get("status", "NOVO") for m in all_monitors}) or ["NOVO"]
    a, b, c = st.columns(3)
    priority = a.multiselect("Prioridade", priorities, default=priorities)
    status = b.multiselect("Status", statuses, default=statuses)
    units = sorted({m.get("unit_external_id") for m in all_monitors if m.get("unit_external_id")})
    unit = c.selectbox("Unidade", ["Todas", *units])
    search = st.text_input("Criança ou identificador")
    visible = [m for m in all_monitors if m.get("priority") in priority and m.get("status") in status
               and (unit == "Todas" or m.get("unit_external_id") == unit)
               and (not search or search.casefold() in str(m.get("child_external_id", "")).casefold() or search.casefold() in m.get("description", "").casefold())]
    table = [{"Criança": m.get("child_external_id"), "Unidade": m.get("unit_external_id") or "Sem informação", "Pendência": m.get("description"), "Prioridade": m.get("priority"), "Status": m.get("status"), "Identificada em": m.get("identified_at")} for m in visible]
    st.dataframe(pd.DataFrame(table), use_container_width=True, hide_index=True)
    if visible:
        labels = {str(m["_id"]): f"{m.get('child_external_id')} · {m.get('type')} · {m.get('status')}" for m in visible}
        selected = st.selectbox("Atualizar ocorrência", list(labels), format_func=lambda value: labels[value])
        monitor = next(m for m in visible if str(m["_id"]) == selected)
        with st.form("monitor_update"):
            states = ["NOVO", "EM_ACOMPANHAMENTO", "PENDENTE", "RESOLVIDO", "DESCARTADO"]
            new_status = st.selectbox("Novo status", states, index=states.index(monitor.get("status", "NOVO")))
            note = st.text_area("Observação", value=monitor.get("note") or "")
            reason = st.text_input("Justificativa para resolver/descartar")
            responsible = st.text_input("Profissional responsável", value=monitor.get("responsible") or "")
            if st.form_submit_button("Salvar atualização"):
                if role not in ("ADMIN", "PROFISSIONAL"):
                    st.error("Seu perfil não pode alterar ocorrências.")
                else:
                    try:
                        update_monitoring(db, selected, {"status": new_status, "note": note or None, "resolution_reason": reason or None, "responsible": responsible or None}, user["username"])
                        st.success("Histórico atualizado.")
                        st.rerun()
                    except Exception as exc:
                        st.error(str(exc))
    if not visible:
        st.info("Nenhuma ocorrência corresponde aos filtros.")
    if role == "ADMIN":
        with st.expander("Regras operacionais e intervalos aprovados"):
            st.markdown("**Triagens**")
            for key, config in SCREENINGS.items():
                rule_id = f"triagem_sem_informacao_{key}"
                rule = db.regras_monitoramento.find_one({"rule_id": rule_id}) or {}
                c1, c2, c3 = st.columns([3, 2, 1])
                c1.write(config["label"])
                enabled = c2.checkbox("Ativa", value=rule.get("enabled", True), key=f"enabled_{key}")
                options = ["BAIXA", "MEDIA", "ALTA", "CRITICA"]
                priority_value = c3.selectbox("Prioridade", options, index=options.index(rule.get("priority", "BAIXA")), key=f"priority_{key}")
                if st.button("Salvar regra", key=f"save_rule_{key}"):
                    configure_rule(db, rule_id, enabled, priority_value)
                    audit(db, user["username"], "regra_atualizada", "regra_monitoramento", rule_id, {"enabled": enabled, "priority": priority_value})
                    st.success("Regra salva.")
            st.markdown("**Acompanhamento temporal**")
            st.caption("Informe prazos aprovados pelo serviço. Não existe valor clínico padrão e as regras começam desligadas.")
            for rule_id, (occurrence_type, description) in TEMPORAL_RULES.items():
                rule = db.regras_monitoramento.find_one({"rule_id": rule_id}) or {}
                c1, c2, c3, c4 = st.columns([2, 1, 2, 1])
                c1.write(description)
                enabled = c2.checkbox("Ativa", value=rule.get("enabled", False), key=f"temporal_enabled_{rule_id}")
                days = c3.number_input("Intervalo validado (dias)", min_value=0, max_value=3650,
                    value=int(rule.get("threshold_days") or 0), step=1, key=f"temporal_days_{rule_id}")
                options = ["BAIXA", "MEDIA", "ALTA", "CRITICA"]
                priority_value = c4.selectbox("Prioridade", options, index=options.index(rule.get("priority", "BAIXA")), key=f"temporal_priority_{rule_id}")
                if st.button("Salvar intervalo", key=f"save_temporal_{rule_id}"):
                    try:
                        configure_rule(db, rule_id, enabled, priority_value, int(days) if days else None)
                        audit(db, user["username"], "regra_temporal_atualizada", "regra_monitoramento", rule_id,
                              {"enabled": enabled, "threshold_days": int(days) if days else None, "priority": priority_value})
                        st.success("Regra salva. Valide o intervalo com a equipe responsável.")
                    except ValueError as exc:
                        st.error(str(exc))

def render_children():
    st.title("Crianças recebidas")
    search = st.text_input("Pesquisar nome ou identificador")
    visible = [c for c in all_children if not search or search.casefold() in child_name(c).casefold() or search.casefold() in c.get("external_id", "").casefold()]
    st.dataframe(pd.DataFrame([{"Identificação": child_name(c), "ID de origem": c.get("external_id"), "Nascimento": birth_date(c) or "Sem informação", "Idade em dias": age_days(c) if age_days(c) is not None else "—", "Unidade": c.get("unit_external_id") or "Sem informação", "Atualizado": c.get("updated_at")} for c in visible]), use_container_width=True, hide_index=True)
    if visible:
        selected = st.selectbox("Detalhar criança", [f"{c['source_system']}::{c['external_id']}" for c in visible])
        source, external_id = selected.split("::", 1)
        child = next(c for c in visible if c["source_system"] == source and c["external_id"] == external_id)
        st.subheader("Projeção recebida")
        st.json(json_safe(child.get("projections", {})))
        events = list(db.eventos_integracao.find({"source_system": source, "child_external_id": external_id}).sort("occurred_at", -1))
        st.subheader("Histórico de origem")
        st.dataframe(pd.DataFrame([{"Tipo": e.get("event_type"), "Evento": e.get("event_id"), "Ocorrido em": e.get("occurred_at"), "Versão": e.get("version")} for e in events]), use_container_width=True, hide_index=True)
        monitors = list(db.monitoramentos.find({"source_system": source, "child_external_id": external_id}).sort("updated_at", -1))
        st.subheader("Monitoramentos")
        st.dataframe(pd.DataFrame([{"Tipo": m.get("type"), "Estado dos dados": m.get("data_state"), "Prioridade": m.get("priority"), "Status": m.get("status"), "Histórico": len(m.get("history", []))} for m in monitors]), use_container_width=True, hide_index=True)

def summary_pdf(summary):
    buffer = BytesIO()
    doc = SimpleDocTemplate(buffer, pagesize=A4, title="Resumo digital de alta")
    styles = getSampleStyleSheet()
    story = [Paragraph("Resumo Digital de Alta da Maternidade", styles["Title"]), Spacer(1, 10)]
    for label, value in (("Documento", summary.get("document_id")), ("Gerado em", summary.get("generated_at")), ("Estado", summary.get("status")), ("Origem", summary.get("source_system"))):
        story.append(Paragraph(f"{label}: {html.escape(str(value or 'Sem informação'))}", styles["Normal"]))
    story.append(Paragraph("Fonte: eventos recebidos do sistema de origem. Campo não recebido aparece como sem informação.", styles["Normal"]))
    for section, content in summary.get("sections", {}).items():
        story.extend([Spacer(1, 12), Paragraph(html.escape(section.replace("_", " ").title()), styles["Heading2"])])
        fields = content.items() if isinstance(content, dict) else [("Registro", item) for item in content] if isinstance(content, list) else [("Informação", content)]
        rows = [[html.escape(str(key).replace("_", " ").title()), html.escape(json.dumps(value, ensure_ascii=False, default=str)) if value not in (None, "") else "Sem informação disponível"] for key, value in fields]
        if not rows:
            rows = [["Informação", "Sem informação disponível"]]
        table = Table(rows, colWidths=[160, 340], repeatRows=0)
        table.setStyle(TableStyle([("BACKGROUND", (0,0), (-1,0), colors.HexColor("#d8eee9")), ("GRID", (0,0), (-1,-1), .35, colors.HexColor("#9bbab6")), ("VALIGN", (0,0), (-1,-1), "TOP"), ("PADDING", (0,0), (-1,-1), 6)]))
        story.append(table)
    if summary.get("missing_fields"):
        story.extend([Spacer(1, 10), Paragraph("Campos sem informação: " + html.escape(", ".join(summary["missing_fields"])), styles["Normal"])])
    story.extend([Spacer(1, 20), Paragraph("Documento de transição do cuidado. Não substitui prontuário ou avaliação profissional. Não possui assinatura digital legal.", styles["Italic"])])
    doc.build(story)
    return buffer.getvalue()

def render_discharge():
    st.title("Resumo digital de alta")
    if not all_children:
        st.info("Aguardando eventos de nascimento/alta enviados pela origem.")
        return
    labels = {f"{c['source_system']}::{c['external_id']}": f"{child_name(c)} · {c['external_id']}" for c in all_children}
    child_key = st.selectbox("Selecionar criança", list(labels), format_func=lambda value: labels[value])
    source, external_id = child_key.split("::", 1)
    if st.button("Gerar rascunho do resumo", type="primary"):
        try:
            create_discharge_summary(db, source, external_id, user["username"])
            st.rerun()
        except Exception as exc:
            st.error(str(exc))
    summaries = list(db.resumos_alta.find({"source_system": source, "child_external_id": external_id}).sort("generated_at", -1))
    if not summaries:
        st.info("Nenhum resumo gerado.")
        return
    summary = summaries[0]
    st.caption(f"{summary['document_id']} · {summary['status']} · {summary['generated_at']}")
    st.json(json_safe(summary.get("sections", {})))
    if summary.get("missing_fields"):
        st.warning("Sem informação disponível: " + ", ".join(summary["missing_fields"]))
    if role in ("ADMIN", "PROFISSIONAL") and summary.get("status") == "RASCUNHO":
        if st.button("Enviar para validação"):
            now = datetime.now(timezone.utc)
            db.resumos_alta.update_one({"_id": summary["_id"]}, {"$set": {"status": "PENDENTE_VALIDACAO", "updated_at": now}})
            audit(db, user["username"], "resumo_enviado_validacao", "resumo_alta", summary["document_id"])
            st.rerun()
    if role == "PROFISSIONAL" and summary.get("status") in ("PENDENTE_VALIDACAO", "ATUALIZADO"):
        if st.button("Validar resumo como profissional"):
            now = datetime.now(timezone.utc)
            db.resumos_alta.update_one({"_id": summary["_id"]}, {"$set": {"status": "VALIDADO", "validated_by": user["username"], "validated_at": now}})
            audit(db, user["username"], "resumo_validado", "resumo_alta", summary["document_id"], {"validated_at": now.isoformat()})
            st.success("Validação registrada sem simular assinatura digital.")
            st.rerun()
    st.download_button("Baixar PDF", summary_pdf(json_safe(summary)), f"resumo-alta-{summary['document_id']}.pdf", "application/pdf")
    st.subheader("Histórico de geração")
    st.dataframe(pd.DataFrame([{ "Documento": s.get("document_id"), "Estado": s.get("status"), "Gerado em": s.get("generated_at"), "Validado por": s.get("validated_by") or "—" } for s in summaries]), use_container_width=True, hide_index=True)

def render_integration():
    st.title("Integração de dados")
    st.write("A origem relacional envia eventos JSON por HTTPS. Esta aplicação não abre conexão com o SQL Server.")
    st.code("""POST /api/v1/integracao/registros
X-API-Key: <chave configurada separadamente>
Content-Type: application/json

{"event_id":"evt-001","source_system":"sistema_relacional","schema_version":"1.0","event_type":"alta_maternidade","entity_type":"crianca","entity_external_id":"CRI-001","child_external_id":"CRI-001","unit_external_id":"UBS-001","occurred_at":"2026-11-20T14:30:00-03:00","data":{"data_alta":"2026-11-20"}}""", language="json")
    st.caption("OpenAPI: /docs na URL da API. Não compartilhe a chave em capturas ou logs.")
    receipts = list(db.recebimentos_integracao.find().sort("received_at", -1).limit(100))
    st.dataframe(pd.DataFrame([{ "Evento": r.get("event_id", "—"), "Estado": r.get("status"), "Recebido em": r.get("received_at"), "Erro": r.get("error_codes", []) } for r in receipts]), use_container_width=True, hide_index=True)

def render_audit():
    st.title("Auditoria")
    logs = list(db.auditoria.find().sort("created_at", -1).limit(200))
    st.dataframe(pd.DataFrame([{ "Data": item.get("created_at"), "Ator": item.get("actor"), "Ação": item.get("action"), "Entidade": item.get("entity_type"), "ID": item.get("entity_id"), "Detalhes": str(item.get("details", {})) } for item in logs]), use_container_width=True, hide_index=True)
    if role == "ADMIN":
        with st.expander("Criar usuário da plataforma"):
            with st.form("create_user"):
                username = st.text_input("Nome de usuário")
                password = st.text_input("Senha temporária", type="password")
                new_role = st.selectbox("Perfil", ["PROFISSIONAL", "LEITURA"])
                submitted = st.form_submit_button("Criar usuário")
            if submitted:
                if len(password) < 12:
                    st.error("Use senha com pelo menos 12 caracteres.")
                elif db.usuarios.find_one({"username": username}):
                    st.error("Usuário já existe.")
                else:
                    db.usuarios.insert_one({"username": username, "password": hash_password(password), "role": new_role, "active": True, "created_at": datetime.now(timezone.utc)})
                    audit(db, user["username"], "usuario_criado", "usuario", username, {"role": new_role})
                    st.success("Usuário criado.")

{"Visão geral": render_dashboard, "Inteligência de dados": render_intelligence,
 "Central de monitoramento": render_monitoring, "Crianças": render_children,
 "Resumo digital de alta": render_discharge, "Integração": render_integration,
 "Auditoria": render_audit}[page]()
''', encoding="utf-8")
print("Interface Streamlit gerada com filtros, indicadores, monitoramento, PDFs e auditoria.")

In [ ]:
# Célula 9.1 — Acompanhamentos em aberto por unidade
# Complementa o gráfico de status/prioridade e usa a API atual do Streamlit.
import ast
app_path = APP_HOME / "app.py"
source = app_path.read_text(encoding="utf-8")
chart = '                st.plotly_chart(px.histogram(monitor_frame, x="Status", color="Prioridade", title="Monitoramentos por status/prioridade", color_discrete_sequence=PALETTE), use_container_width=True)'
if source.count(chart) != 1:
    raise RuntimeError("Não foi possível localizar o gráfico de monitoramento para incluir a visão por unidade.")
source = source.replace(chart, chart + '\n                pending_frame = monitor_frame[monitor_frame["Status"].isin(["NOVO", "PENDENTE", "EM_ACOMPANHAMENTO"])]\n                if not pending_frame.empty:\n                    st.plotly_chart(px.histogram(pending_frame, x="Unidade", color="Prioridade", title="Acompanhamentos em aberto por unidade", color_discrete_sequence=PALETTE), use_container_width=True)')
source = source.replace("use_container_width=True", 'width="stretch"')
ast.parse(source, filename="app.py")
app_path.write_text(source, encoding="utf-8")
print("Gráfico de pendências por unidade incluído; propriedades Streamlit atualizadas.")

In [ ]:
# Célula 10 — Inicialização da API e do Streamlit
# Os processos usam variáveis de ambiente em memória; logs não registram cabeçalhos nem segredos.
import os
import subprocess
import sys
import time
import requests

runtime_env = {**os.environ, "PYTHONPATH": str(APP_HOME)}
api_log = open("/tmp/saude_infantil_api.log", "a", encoding="utf-8")
ui_log = open("/tmp/saude_infantil_streamlit.log", "a", encoding="utf-8")
API_PROCESS = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "api:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd=APP_HOME, env=runtime_env, stdout=api_log, stderr=subprocess.STDOUT, start_new_session=True,
)
STREAMLIT_PROCESS = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", "app.py", "--server.address", "0.0.0.0", "--server.port", "8501", "--server.headless", "true", "--browser.gatherUsageStats", "false"],
    cwd=APP_HOME, env=runtime_env, stdout=ui_log, stderr=subprocess.STDOUT, start_new_session=True,
)
API_LOCAL_URL = "http://127.0.0.1:8000"
STREAMLIT_LOCAL_URL = "http://127.0.0.1:8501"
for _ in range(40):
    try:
        health = requests.get(f"{API_LOCAL_URL}/health", timeout=2)
        if health.ok:
            break
    except requests.RequestException:
        pass
    if API_PROCESS.poll() is not None:
        raise RuntimeError("A API encerrou ao iniciar. Consulte /tmp/saude_infantil_api.log.")
    time.sleep(1)
else:
    raise TimeoutError("API não respondeu ao health check; confira MongoDB e o log local.")
print("API e Streamlit iniciados. A API respondeu ao health check local.")
print("Documentação local da API: http://127.0.0.1:8000/docs")

In [ ]:
# Célula 11 — Túneis HTTP Cloudflare para demonstração
# URLs efêmeras, públicas e diferentes a cada execução. Use somente dados sintéticos.
import os
import platform
import re
import select
import subprocess
import time
import urllib.request
from pathlib import Path

if platform.machine().lower() not in ("x86_64", "amd64"):
    raise RuntimeError("Esta célula baixa o binário amd64 do cloudflared, esperado no runtime padrão do Colab.")
CLOUDFLARED = Path("/tmp/cloudflared")
if not CLOUDFLARED.exists():
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", CLOUDFLARED)
    CLOUDFLARED.chmod(0o755)

def start_tunnel(port):
    process = subprocess.Popen([str(CLOUDFLARED), "tunnel", "--url", f"http://127.0.0.1:{port}"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    deadline = time.time() + 40
    while time.time() < deadline:
        ready, _, _ = select.select([process.stdout], [], [], 1)
        if ready:
            line = process.stdout.readline()
            match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
            if match:
                return process, match.group(0)
        if process.poll() is not None:
            raise RuntimeError("cloudflared encerrou antes de criar a URL.")
    process.terminate()
    raise TimeoutError("O túnel não gerou uma URL em 40 segundos.")

API_TUNNEL_PROCESS, API_PUBLIC_URL = start_tunnel(8000)
STREAMLIT_TUNNEL_PROCESS, STREAMLIT_PUBLIC_URL = start_tunnel(8501)
os.environ["PUBLIC_API_URL"] = API_PUBLIC_URL
print(f"Interface Streamlit: {STREAMLIT_PUBLIC_URL}")
print(f"API (protegida por X-API-Key): {API_PUBLIC_URL}")
print(f"OpenAPI: {API_PUBLIC_URL}/docs")
print("As URLs públicas mudam a cada execução. Encerre os processos ao terminar a demonstração.")

In [ ]:
# Célula 12 — Eventos de demonstração sintéticos
# IDs fixos tornam a execução repetível e idempotente. Nenhum dado corresponde a paciente real.
from datetime import datetime, timedelta, timezone
import requests

now = datetime.now(timezone.utc)
birth_day = (now - timedelta(days=7)).date().isoformat()
demo_events = [
    {"event_id": "demo-001-nascimento", "source_system": "demo_sintetico", "schema_version": "1.0", "event_type": "nascimento", "entity_type": "crianca", "entity_external_id": "DEMO-001", "child_external_id": "DEMO-001", "unit_external_id": "UBS-DEMO-01", "occurred_at": f"{birth_day}T09:00:00+00:00", "data": {"nome": "Criança Sintética A", "data_nascimento": birth_day, "sexo": "F", "peso_nascimento": 3120, "comprimento_nascimento": 49}},
    {"event_id": "demo-001-alta", "source_system": "demo_sintetico", "schema_version": "1.0", "event_type": "alta_maternidade", "entity_type": "crianca", "entity_external_id": "DEMO-001", "child_external_id": "DEMO-001", "unit_external_id": "UBS-DEMO-01", "occurred_at": (now - timedelta(days=5)).isoformat(), "data": {"data_alta": (now - timedelta(days=5)).date().isoformat(), "unidade_hospitalar": "Maternidade Sintética", "orientacoes": "Acompanhamento conforme avaliação da equipe."}},
    {"event_id": "demo-001-triagem-pe", "source_system": "demo_sintetico", "schema_version": "1.0", "event_type": "triagem", "entity_type": "crianca", "entity_external_id": "DEMO-001", "child_external_id": "DEMO-001", "unit_external_id": "UBS-DEMO-01", "occurred_at": (now - timedelta(days=6)).isoformat(), "data": {"tipo_triagem": "pezinho", "status": "realizado", "data_realizacao": (now - timedelta(days=6)).date().isoformat()}},
    {"event_id": "demo-002-nascimento", "source_system": "demo_sintetico", "schema_version": "1.0", "event_type": "nascimento", "entity_type": "crianca", "entity_external_id": "DEMO-002", "child_external_id": "DEMO-002", "unit_external_id": "UBS-DEMO-02", "occurred_at": f"{(now - timedelta(days=3)).date().isoformat()}T11:00:00+00:00", "data": {"nome": "Criança Sintética B", "data_nascimento": (now - timedelta(days=3)).date().isoformat(), "sexo": "M"}},
    {"event_id": "demo-001-vacinacao", "source_system": "demo_sintetico", "schema_version": "1.0", "event_type": "vacinacao", "entity_type": "crianca", "entity_external_id": "DEMO-001", "child_external_id": "DEMO-001", "unit_external_id": "UBS-DEMO-01", "occurred_at": (now - timedelta(days=4)).isoformat(), "data": {"vacina": "Vacina sintética", "dose": "Demonstração", "status": "registrada"}},
]
headers = {"X-API-Key": API_INTEGRATION_KEY}
for event in demo_events:
    response = requests.post(f"{API_LOCAL_URL}/api/v1/integracao/registros", json=event, headers=headers, timeout=15)
    if response.status_code not in (200, 202):
        raise RuntimeError(f"Evento sintético rejeitado ({response.status_code}): {response.text[:300]}")
print(f"{len(demo_events)} eventos sintéticos enviados pela API; repetições não duplicam os registros.")

In [ ]:
# Célula 13 — Testes sintéticos automatizados
# Isola o Mongo com mongomock: esta célula não escreve na base Atlas configurada.
(APP_HOME / "test_platform.py").write_text(r'''import ast
import html
import json
import os
import sys
from datetime import datetime, timedelta, timezone
from io import BytesIO
import mongomock
import pytest
from fastapi.testclient import TestClient
from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle

sys.path.insert(0, os.environ["APP_HOME"])
os.environ["API_INTEGRATION_KEY"] = "test-key-not-a-real-secret"
import api
from database import ensure_indexes
from monitoring import ensure_default_rules, refresh_child_monitorings, configure_rule
from services import process_event, create_discharge_summary, update_monitoring

db = mongomock.MongoClient().inteligencia_teste
ensure_indexes(db)
ensure_default_rules(db)
api.get_db = lambda: db
api.app.router.on_startup.clear()
client = TestClient(api.app)


def event(event_id="test-birth", data=None, occurred_at=None, event_type="nascimento", child_id="TEST-001"):
    return {
        "event_id": event_id, "source_system": "teste_sintetico", "schema_version": "1.0",
        "event_type": event_type, "entity_type": "crianca", "entity_external_id": child_id,
        "child_external_id": child_id, "unit_external_id": "UNIT-TEST",
        "occurred_at": occurred_at or "2026-11-24T12:00:00+00:00",
        "data": data or {"nome": "Criança Sintética", "data_nascimento": "2026-11-20", "peso_nascimento": 3000},
    }


def test_api_rejects_missing_integration_key_and_invalid_contract():
    assert client.post("/api/v1/integracao/registros", json=event()).status_code == 401
    invalid = client.post("/api/v1/integracao/registros", json={"source_system": "x"}, headers={"X-API-Key": "test-key-not-a-real-secret"})
    assert invalid.status_code == 422 and invalid.json()["error"]["code"] == "validation_error"


def test_batch_and_event_lookup():
    headers = {"X-API-Key": "test-key-not-a-real-secret"}
    response = client.post("/api/v1/integracao/registros/lote", json={"events": [event(event_id="batch-event")]}, headers=headers)
    assert response.status_code == 202 and response.json()["received"] == 1
    lookup = client.get("/api/v1/integracao/registros/batch-event?source_system=teste_sintetico", headers=headers)
    assert lookup.status_code == 200 and lookup.json()["data"]["event_id"] == "batch-event"


def test_idempotency_correction_and_monitor_note_preservation():
    headers = {"X-API-Key": "test-key-not-a-real-secret"}
    first = client.post("/api/v1/integracao/registros", json=event(), headers=headers)
    duplicate = client.post("/api/v1/integracao/registros", json=event(), headers=headers)
    assert first.status_code == duplicate.status_code == 202
    assert duplicate.json()["data"]["duplicate"] is True
    assert db.eventos_integracao.count_documents({"event_id": "test-birth"}) == 1
    monitor = db.monitoramentos.find_one({"child_external_id": "TEST-001", "occurrence_key": "pezinho"})
    db.monitoramentos.update_one({"_id": monitor["_id"]}, {"$set": {"note": "Anotação preservada"}})
    corrected = event(data={"nome": "Criança Sintética", "data_nascimento": "2026-11-20", "peso_nascimento": 3100})
    result = client.post("/api/v1/integracao/registros", json=corrected, headers=headers)
    assert result.json()["data"]["status"] == "CORRIGIDO"
    child = db.criancas.find_one({"external_id": "TEST-001"})
    assert child["projections"]["nascimento"]["data"]["peso_nascimento"] == 3100
    assert db.monitoramentos.find_one({"_id": monitor["_id"]})["note"] == "Anotação preservada"


def test_monitoring_status_updates_keep_notes_and_audit_history():
    monitor = db.monitoramentos.find_one({"child_external_id": "TEST-001", "occurrence_key": "pezinho"})
    updated = update_monitoring(db, str(monitor["_id"]), {"status": "EM_ACOMPANHAMENTO", "note": "Contato solicitado", "responsible": "Profissional sintético"}, "tester")
    resolved = update_monitoring(db, str(monitor["_id"]), {"status": "RESOLVIDO", "resolution_reason": "Origem confirmou atualização."}, "tester")
    assert updated["note"] == resolved["note"] == "Contato solicitado"
    assert resolved["status"] == "RESOLVIDO" and resolved["resolution_reason"]
    assert len(resolved["history"]) >= 3
    assert db.auditoria.count_documents({"entity_id": str(monitor["_id"])}) >= 2


def test_out_of_order_event_does_not_replace_newer_projection():
    process_event(db, event(event_id="new-birth", data={"nome": "Criança Sintética", "data_nascimento": "2026-11-20", "peso_nascimento": 3100}))
    older = event(event_id="old-birth", data={"peso_nascimento": 2500}, occurred_at="2026-11-19T10:00:00+00:00")
    assert process_event(db, older)["projection_status"] == "FORA_DE_ORDEM"
    assert db.criancas.find_one({"external_id": "TEST-001"})["projections"]["nascimento"]["data"]["peso_nascimento"] == 3100


def test_missing_triage_is_labeled_as_missing_information():
    process_event(db, event(event_id="triage-child-birth", child_id="MISSING-TRIAGE-01"))
    monitors = list(db.monitoramentos.find({"child_external_id": "MISSING-TRIAGE-01", "type": "TRIAGEM_SEM_INFORMACAO"}))
    assert len(monitors) == 4
    assert all(item["data_state"] == "SEM_INFORMACAO" for item in monitors)
    assert all("não confirma não realização" in item["description"] for item in monitors)


def test_explicit_not_performed_is_not_mislabeled_as_missing():
    process_event(db, event(event_id="explicit-birth", child_id="EXPLICIT-01"))
    process_event(db, event(event_id="explicit-triage", event_type="triagem", child_id="EXPLICIT-01", data={"tipo_triagem": "pezinho", "status": "não realizado"}))
    confirmed = db.monitoramentos.find_one({"child_external_id": "EXPLICIT-01", "occurrence_key": "pezinho:nao_realizado"})
    missing = db.monitoramentos.find_one({"child_external_id": "EXPLICIT-01", "occurrence_key": "pezinho"})
    assert confirmed["type"] == "TRIAGEM_NAO_REALIZADA"
    assert confirmed["data_state"] == "NAO_REALIZADO_CONFIRMADO_PELA_ORIGEM"
    assert missing["status"] == "RESOLVIDO"


def test_temporal_rules_stay_disabled_until_configured_and_resolve_on_followup():
    rule = db.regras_monitoramento.find_one({"rule_id": "alta_sem_consulta"})
    assert rule["enabled"] is False and rule["threshold_days"] is None
    with pytest.raises(ValueError):
        configure_rule(db, "alta_sem_consulta", True, "BAIXA")
    child_id = "TEMPORAL-01"
    process_event(db, event(event_id="temporal-birth", child_id=child_id))
    process_event(db, event(event_id="temporal-discharge", event_type="alta_maternidade", child_id=child_id, data={"data_alta": "2026-11-20"}))
    old = datetime.now(timezone.utc) - timedelta(days=2)
    db.criancas.update_one({"external_id": child_id}, {"$set": {
        "updated_at": old, "projections.alta_maternidade.occurred_at": old,
    }})
    configure_rule(db, "alta_sem_consulta", True, "BAIXA", threshold_days=1)
    refresh_child_monitorings(db, "teste_sintetico", child_id)
    occurrence = db.monitoramentos.find_one({"child_external_id": child_id, "rule_id": "alta_sem_consulta"})
    assert occurrence["status"] == "PENDENTE" and occurrence["threshold_days"] == 1
    process_event(db, event(event_id="temporal-follow-up", event_type="consulta", child_id=child_id, data={"data_consulta": datetime.now(timezone.utc).isoformat()}))
    assert db.monitoramentos.find_one({"_id": occurrence["_id"]})["status"] == "RESOLVIDO"


def test_summary_and_pdf_do_not_invent_missing_clinical_data():
    summary = create_discharge_summary(db, "teste_sintetico", "TEST-001", "tester")
    assert summary["status"] == "RASCUNHO" and "peso_nascimento" in summary["sections"]["nascimento"]
    assert "teste de olhinho" not in str(summary["sections"]).lower()
    source = open(os.path.join(os.environ["APP_HOME"], "app.py"), encoding="utf-8").read()
    tree = ast.parse(source)
    function = next(node for node in tree.body if isinstance(node, ast.FunctionDef) and node.name == "summary_pdf")
    namespace = {"html": html, "json": json, "BytesIO": BytesIO, "SimpleDocTemplate": SimpleDocTemplate,
                 "A4": A4, "getSampleStyleSheet": getSampleStyleSheet, "Paragraph": Paragraph,
                 "Spacer": Spacer, "Table": Table, "TableStyle": TableStyle, "colors": colors}
    exec(compile(ast.Module(body=[function], type_ignores=[]), "app.py", "exec"), namespace)
    assert namespace["summary_pdf"](summary).startswith(b"%PDF")
    process_event(db, event(event_id="summary-source-correction", data={"peso_nascimento": 3050}))
    assert db.resumos_alta.find_one({"document_id": summary["document_id"]})["status"] == "ATUALIZADO"


def test_health_responds_without_exposing_credentials():
    response = client.get("/health")
    assert response.status_code == 200 and response.json()["ok"] is True
    assert "MONGO_URI" not in response.text and "API_INTEGRATION_KEY" not in response.text
''', encoding="utf-8")
import pytest
result = pytest.main(["-q", str(APP_HOME / "test_platform.py")])
if result != 0:
    raise RuntimeError(f"Testes falharam (código {result}).")
print("Testes executados em Mongo simulado, sem usar dados clínicos nem o Atlas.")